# Gemma-2 CBT Anxiety Support: train, serve, evaluate

**Part A** (any GPU runtime, T4 works): prepare CACTUS data, QLoRA fine-tune Gemma-2-2B-it, measure base vs fine-tuned perplexity, save the adapter to Google Drive.

**Part B** (fresh runtime, L4 / A100 recommended): serve Gemma-2 + the adapter and Llama Guard 3 with vLLM, start the FastAPI service, run the full-stack safety and response-quality evaluations.

Before you start:
1. Accept the licences on Hugging Face for `google/gemma-2-2b-it` and `meta-llama/Llama-Guard-3-1B`.
2. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) with a read token.

## Part A: fine-tune

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/comet-web/gemma2-anxiety-support.git"
REPO_DIR = "/content/repo"
# Safe to re-run: clones once, then just pulls the latest commit.
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
os.chdir(REPO_DIR)
assert os.path.isfile("pyproject.toml"), f"repo not found in {os.getcwd()}"
print("working dir:", os.getcwd())
!pip install -q -e ".[train,dev]"
# Colab ships torchao 0.10; PEFT >= 0.21 refuses torchao < 0.16 and this project does not use it
!pip uninstall -y -q torchao

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(os.environ["HF_TOKEN"])

In [ ]:
# ~13k anxiety-focused CBT dialogues -> Gemma-2 chat format, train/eval/test split
!python training/prepare_data.py

Training time depends on the GPU. `max_train_samples: 4000` in `configs/train_qlora.yaml` is a few hours on a T4 and
much faster on an L4 / A100. For a quick first run pass `--max-train-samples 1000`.

In [ ]:
# 1) ~1 minute pipeline check on a tiny random Gemma-2: catches config/library errors before the long run
!python training/train_qlora.py --smoke
# 2) the real QLoRA run (add --max-train-samples 1000 for a quicker first pass)
!python training/train_qlora.py --config configs/train_qlora.yaml


In [ ]:
import json
m = json.load(open("evaluation/results/training_metrics.json"))
print({k: v for k, v in m.items() if k != "config"})

In [ ]:
# Qualitative check with transformers + PEFT (base model vs fine-tuned adapter on the same prompt)
import sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
sys.path.insert(0, "src")
from anxiety_support.prompts import COUNSELOR_SYSTEM_PROMPT

base_id = "google/gemma-2-2b-it"
tok = AutoTokenizer.from_pretrained(base_id)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4 -> fp16
base = AutoModelForCausalLM.from_pretrained(base_id, dtype=dtype, device_map="auto")
model = PeftModel.from_pretrained(base, "outputs/gemma2-cbt-qlora/adapter")
model.eval()

msgs = [{"role": "user", "content": COUNSELOR_SYSTEM_PROMPT + "

I have a presentation tomorrow and I'm sure everyone will think I'm stupid."}]
# transformers v5: apply_chat_template returns a BatchEncoding (dict), not a bare tensor
inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    print("FINE-TUNED:
", tok.decode(out[0, prompt_len:], skip_special_tokens=True))
    with model.disable_adapter():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    print("
BASE:
", tok.decode(out[0, prompt_len:], skip_special_tokens=True))

del model, base
torch.cuda.empty_cache()


In [ ]:
# Persist the adapter + metrics to Drive for Part B
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/gemma2-cbt
!cp -r outputs/gemma2-cbt-qlora/adapter /content/drive/MyDrive/gemma2-cbt/
!cp evaluation/results/training_metrics.json /content/drive/MyDrive/gemma2-cbt/
# Optional: push the adapter to the Hub (private)
# !python training/merge_adapter.py --adapter outputs/gemma2-cbt-qlora/adapter --push-adapter <hf-user>/gemma2-2b-cbt-lora

## Part B: serve with vLLM and evaluate the full stack

**Runtime > Disconnect and delete runtime**, then start a fresh one (vLLM brings its own torch build) and run from here.

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/comet-web/gemma2-anxiety-support.git"
REPO_DIR = "/content/repo"
# Safe to re-run: clones once, then just pulls the latest commit.
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
os.chdir(REPO_DIR)
assert os.path.isfile("pyproject.toml"), f"repo not found in {os.getcwd()}"
print("working dir:", os.getcwd())
!pip install -q vllm==0.32.0
!pip install -q -e ".[dev]"
import os
from google.colab import userdata, drive
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
drive.mount("/content/drive")
!mkdir -p outputs/gemma2-cbt-qlora && cp -r /content/drive/MyDrive/gemma2-cbt/adapter outputs/gemma2-cbt-qlora/
!mkdir -p evaluation/results && cp /content/drive/MyDrive/gemma2-cbt/training_metrics.json evaluation/results/
!python training/prepare_data.py   # recreates the held-out test split (same seed)

In [ ]:
# --enforce-eager: skip torch.compile/CUDA-graph capture (slow and memory-hungry on a T4)
# Llama Guard 3 on :8001, Gemma-2 base + CBT LoRA on :8000 (one GPU, split memory)
!nohup vllm serve meta-llama/Llama-Guard-3-1B --port 8001 --gpu-memory-utilization 0.25 --max-model-len 4096 --enforce-eager > guard.log 2>&1 &
!sleep 5
!nohup vllm serve google/gemma-2-2b-it --port 8000 --enable-lora --lora-modules gemma-cbt=outputs/gemma2-cbt-qlora/adapter --max-lora-rank 16 --gpu-memory-utilization 0.55 --max-model-len 4096 --enforce-eager > llm.log 2>&1 &

import time, urllib.request
for port in (8001, 8000):
    for _ in range(120):
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models"); print(port, "ready"); break
        except Exception:
            time.sleep(5)
    else:
        raise RuntimeError(f"vLLM on {port} did not start, see guard.log / llm.log")

In [ ]:
!nohup uvicorn anxiety_support.api.main:app --port 8080 > api.log 2>&1 &
!sleep 20
!curl -s localhost:8080/v1/chat -H 'content-type: application/json' -d '{"message": "I keep panicking before exams and my mind goes blank"}' | python -m json.tool

In [ ]:
# Safety ablation on the held-out prompt set
!python evaluation/eval_safety.py --target guard --split test
!python evaluation/eval_safety.py --target api --split test

In [ ]:
# Base vs fine-tuned response quality on held-out CACTUS conversations.
# Add --judge-model/--judge-base-url/--judge-api-key to use a stronger LLM judge (pairwise, both orderings).
!python evaluation/eval_responses.py --base-model google/gemma-2-2b-it --ft-model gemma-cbt -n 100

In [ ]:
# Save results so you can commit them to the repo
!cp -r evaluation/results /content/drive/MyDrive/gemma2-cbt/
!ls /content/drive/MyDrive/gemma2-cbt/results